In [ ]:
!pip install rasterio torch torchvision tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
!apt-get install tree -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 0s (355 kB/s)
Selecting previously unselected package tree.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!tree /content/drive/MyDrive -L 1

/content/drive/MyDrive
├── 2026_1
├── CECiD
├── Colab Notebooks
└── Q2

4 directories, 0 files


In [ ]:
!tree /content/drive/MyDrive/2026_1/PICD/dataset -L 1

/content/drive/MyDrive/2026_1/PICD/dataset
├── inputs
└── masks

2 directories, 0 files


In [ ]:
!cp -r "/content/drive/MyDrive/2026_1/PICD/dataset" /content/
DATASET_ROOT = "/content/dataset"

In [ ]:
#DATASET_ROOT = "/content/drive/MyDrive/2026_1/PICD/dataset"

In [ ]:
import os
import numpy as np
import rasterio
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [ ]:

class FireDataset(Dataset):
    def __init__(self, root):
        self.root = root

        # ✔ rutas correctas según tu Drive
        self.inputs_dir = os.path.join(root, "inputs")
        self.masks_dir = os.path.join(root, "masks")

        # ✔ lista de archivos (asume mismo nombre en inputs y masks)
        self.files = sorted(os.listdir(self.inputs_dir))

    def __len__(self):
        return len(self.files)

    def _load_tif(self, path):
        with rasterio.open(path) as src:
            return src.read().astype(np.float32)

    def __getitem__(self, idx):
        fname = self.files[idx]

        # ✔ paths correctos en Drive
        x_path = os.path.join(self.inputs_dir, fname)
        y_path = os.path.join(self.masks_dir, fname)

        x = self._load_tif(x_path)   # (25, H, W)
        y = self._load_tif(y_path)   # (1, H, W)

        # ✔ reshape temporal (5 tiempos × 5 variables)
        x = x.reshape(5, 5, x.shape[1], x.shape[2])

        # ✔ normalización estable
        x_min = x.min()
        x_max = x.max()
        x = (x - x_min) / (x_max - x_min + 1e-6)

        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)

        return x, y


In [ ]:
DATASET_ROOT = "/content/drive/MyDrive/2026_1/PICD/dataset"

dataset = FireDataset(DATASET_ROOT)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


class TemporalFireNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = ConvBlock(5, 32)
        self.lstm = nn.LSTM(input_size=32, hidden_size=64, batch_first=True)
        self.decoder = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: (B, T, C, H, W)
        B, T, C, H, W = x.shape

        feats = []

        for t in range(T):
            ft = self.encoder(x[:, t])   # (B, 32, H, W)
            ft = ft.mean(dim=[2,3])      # global pooling -> (B, 32)
            feats.append(ft)

        feats = torch.stack(feats, dim=1)  # (B, T, 32)

        out, _ = self.lstm(feats)          # (B, T, 64)

        last = out[:, -1]                  # (B, 64)

        # expand spatially
        last = last[:, :, None, None].expand(-1, -1, H, W)

        return self.decoder(last)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

#dataset = FireDataset("/content/dataset")
loader = DataLoader(dataset, batch_size=2, shuffle=True)

model = TemporalFireNet().to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    model.train()
    total_loss = 0

    for x, y in tqdm(loader):
        x, y = x.to(device), y.to(device)

        pred = model(x)
        loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}: loss={total_loss/len(loader):.4f}")

100%|██████████| 1501/1501 [12:26<00:00,  2.01it/s]


Epoch 0: loss=0.0039


100%|██████████| 1501/1501 [11:33<00:00,  2.16it/s]


Epoch 1: loss=0.0000


100%|██████████| 1501/1501 [11:45<00:00,  2.13it/s]


Epoch 2: loss=0.0000


  8%|▊         | 118/1501 [01:01<11:55,  1.93it/s]


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

loader = DataLoader(dataset, batch_size=2, shuffle=True)

model = TemporalFireNet().to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 🔥 control de entrenamiento
epoch = 0
max_epochs = 50  # seguridad anti-loop infinito
best_loss = float("inf")
patience = 2
no_improve = 0

while epoch < max_epochs:

    model.train()
    total_loss = 0

    for x, y in tqdm(loader):
        x, y = x.to(device), y.to(device)

        pred = model(x)
        loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch}: loss={avg_loss:.6f}")

    # 💾 CHECKPOINT SIEMPRE
    torch.save(model.state_dict(),
               f"/content/drive/MyDrive/2026_1/PICD/fire_model_epoch_{epoch}.pth")

    # 💾 BEST MODEL
    if avg_loss < best_loss:
        best_loss = avg_loss
        no_improve = 0

        torch.save(model.state_dict(),
                   "/content/drive/MyDrive/2026_1/PICD/fire_model_best.pth")
    else:
        no_improve += 1

    # 🛑 EARLY STOP por estancamiento o loss ~0
    if avg_loss < 1e-6:
        print("🛑 Early stop: loss ~ 0 (posible colapso o convergencia)")
        break

    if no_improve >= patience:
        print("🛑 Early stop: sin mejora")
        break

    epoch += 1

In [ ]:
model.eval()

x, y = dataset[0]
x = x.unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(x)

pred = pred.cpu().numpy()[0,0]
print(pred.shape)
print(pred.min(), pred.max())

(100, 100)
1.3002236e-07 0.00093996135


In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/2026_1/PICD/fire_model.pth")